# Single-Task vs Multitask LLM Instruction Fine-Tuning

Companion notebook for the To Data & Beyond tutorial. It contrasts the two fine-tuning strategies and implements a resource-conscious single-task FLAN-T5 dialogue-summarization example.

## What this notebook demonstrates

- **Single-task fine-tuning:** specialize one pretrained model for dialogue summarization.
- **Multitask fine-tuning:** train across multiple task families to retain broader instruction-following behavior.
- **Catastrophic forgetting:** specialization may reduce performance on capabilities that are absent from the fine-tuning data.

This notebook defaults to `google/flan-t5-small` and a bounded SAMSum subset so it can run meaningfully in Colab. The article's larger/full-dataset configuration requires substantially more compute.

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

MODEL_ID = "google/flan-t5-small"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}")

## Load FLAN-T5 and SAMSum

SAMSum provides messenger-style dialogues paired with human-written summaries, matching the customer-service summarization structure used in the tutorial.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
dataset = load_dataset("knkarthick/samsum")
dataset

## Inspect one training pair

In [ ]:
example = dataset["train"][0]
print("DIALOGUE:\n", example["dialogue"])
print("\nREFERENCE SUMMARY:\n", example["summary"])

## Establish a pretrained baseline

Generate a summary before fine-tuning so you can compare it with the specialized model later.

In [ ]:
def generate_summary(model, dialogue: str, max_new_tokens: int = 80) -> str:
    prompt = "summarize: " + dialogue
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)

model.to(device)
baseline_summary = generate_summary(model, example["dialogue"])
print(baseline_summary)

## Tokenize the dialogue-summary pairs

The source article's archived Taskmaster example referenced fields that dataset does not expose. This corrected version uses SAMSum's actual `dialogue` and `summary` columns.

In [ ]:
def preprocess_function(examples):
    inputs = ["summarize: " + dialogue for dialogue in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=128,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_source = dataset["train"].shuffle(seed=42)
eval_source = dataset["validation"].shuffle(seed=42)
train_source = train_source.select(range(min(512, len(train_source))))
eval_source = eval_source.select(range(min(128, len(eval_source))))

train_dataset = train_source.map(
    preprocess_function,
    batched=True,
    remove_columns=train_source.column_names,
)
eval_dataset = eval_source.map(
    preprocess_function,
    batched=True,
    remove_columns=eval_source.column_names,
)

## Configure the single-task training run

Thirty update steps make this a demonstrable Colab workflow, not a production-quality training run. Increase the data volume, steps, and evaluation rigor for a real deployment.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-samsum-demo",
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=30,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    max_steps=30,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_total_limit=1,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

## Fine-tune and evaluate

In [ ]:
trainer.train()
evaluation = trainer.evaluate()
print(evaluation)

## Compare the specialized model

Generation quality after a short demonstration run will vary. Use a held-out test set and task-appropriate metrics before drawing conclusions.

In [ ]:
fine_tuned_summary = generate_summary(trainer.model, example["dialogue"])
print("BASELINE:\n", baseline_summary)
print("\nFINE-TUNED:\n", fine_tuned_summary)
print("\nREFERENCE:\n", example["summary"])

## Save the model

The output directory is intentionally not committed to the repository.

In [ ]:
OUTPUT_DIR = "./fine-tuned-flan-t5-samsum"
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## From single-task to multitask training

A multitask version would normalize several datasets into the same `input_text`/`target_text` interface, prefix each input with an explicit instruction, interleave or sample the task datasets deliberately, and evaluate each task separately. Balancing matters: a large task can otherwise dominate the mixture.